In [4]:
from ppopt.mplp_program import MPLP_Program
from ppopt.mpmodel import MPModeler
from ppopt.mp_solvers.solve_mpqp import solve_mpqp, mpqp_algorithm
import numpy as np
from typing import List, Tuple, Callable, Union, Dict
import chaospy as cp
import time
from collections import defaultdict
from numpy.polynomial.legendre import leggauss
import sympy as sp
from sympy.logic.boolalg import BooleanTrue, BooleanFalse
import itertools
from pyomo.environ import *
from scipy.optimize import linprog

## Smolyak quadrature

In [ ]:
def theta_interval_at_point(solution, theta_vector: np.ndarray, max_idx: int = 0, min_idx: int = 1) -> tuple:
    """Given the parametric solution for theta_k and the current 'state' vector (theta_prev + d),
        return the scalar lower and upper bound [t_min, t_max] for this theta_k.

    Args:
        solution (_type_): _description_
        t_vector (np.ndarray): _description_
        max_idx (int, optional): _description_. Defaults to 0.
        min_idx (int, optional): _description_. Defaults to 1.

    Returns:
        tuple: _description_
    """

    theta_vector_aug = np.append(theta_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        theta_min = solution[0]
        theta_max = solution[0]
        return float(theta_min), float(theta_max)
    
    for region in solution.critical_regions:
        if region.is_inside(theta_vector.reshape(-1,1)):
            coefficients = np.concatenate([region.A, region.b], axis=1)[:2, :]
            max_coefficients = coefficients[max_idx]
            min_coefficients = coefficients[min_idx]
            theta_max = float(max_coefficients @ theta_vector_aug)
            theta_min = float(min_coefficients @ theta_vector_aug)
            return theta_min, theta_max
    
    raise ValueError("The provided theta_vector is not inside any critical region of the solution.")

In [ ]:
def map_u_to_theta_and_jacobian(solutions: List, u: np.ndarray, d_vector: np.ndarray) -> tuple:
    """Given the parametric solutions for theta_k and the current 'state' vector (theta_prev + d),
        return the theta_k vector and the Jacobian matrix dtheta/du.
    Args:
        solutions (List): List of parametric solutions for each theta_k.
        u (np.ndarray): 1D arraay of canonical coordinates 
        d_vector (np.ndarray): Current disturbance vector.
    """
    theta_values = []
    jacobian = 1.0

    for k, sol in enumerate(solutions):
        if isinstance(d_vector, np.ndarray):
            theta_vector = np.block([np.array(theta_values), d_vector])
        else:
            theta_vector = np.array(theta_values, dtype=float)

        theta_min, theta_max = theta_interval_at_point(sol, theta_vector)
        length = theta_max - theta_min

        theta_k = 0.5 * length * u[k] + 0.5 * (theta_max + theta_min)
        theta_values.append(theta_k)

        jacobian *= 0.5 * length

    return np.array(theta_values, dtype= float), jacobian

In [2]:
def calculate_stocflexibility_smolyak(solutions: List, level: int, joint_func: Callable[[List[float]], float], d_vector: np.ndarray = None, rule: str = "gaussian") -> float:
    """Compute stochastic flexibility using a Smolyak sparse grid in canonical  u-space

    Args:
        solutions (List): list of solutions for each theta dimension (same structure as in calculate_stocflexibility)
        level (int): Smolyak level (1,2,3,...) controls accuracy & number of points
        joint_func (Callable[[List[float]], float]): callable f(theta_list) -> scalar
        d_vector (np.ndarray, optional):design vector (np.ndarray). Defaults to None.
        rule (str, optional): 1D quadrature rule passed to chaospy (e.g. "gaussian"). Defaults to "gaussian".

    Returns:
        float: _description_
    """

    n_theta = len(solutions)

    dist = cp.J(*[cp.Uniform(-1,1) for _ in range(n_theta)])

    nodes_u, weights_expectation = cp.quadrature.sparse_grid(order = level, dist=dist, rule=rule)

    weights_u = weights_expectation * (2.0 ** n_theta)

    nodes_u = nodes_u.T

    start = time.time()
    stochastic_flexibility = 0.0

    for i in range(nodes_u.shape[0]):
        u_vector = nodes_u[i, :]
        theta_vector, jacobian = map_u_to_theta_and_jacobian(solutions, u_vector, d_vector)
        func_value = joint_func(theta_vector)
        stochastic_flexibility += func_value * jacobian * weights_u[i]

    end = time.time()
    print(f"Smolyak stochastic flexibility computed in {end - start:.4f} seconds.")
    return stochastic_flexibility


## Gaussian Legendre quadrature

In [ ]:
def gauss_legendre_between_bounds(expr_coeffs: np.ndarray, n_gl: int, max_idx: int = 0, min_idx: int = 1):
    """
    Generate n Gauss–Legendre quadrature points and weights between min and max bounds
    defined by two linear expressions.

    Parameters:
        expr_coeffs (np.ndarray): 2xD array. Row 0 = max point co`efficients, Row 1 = min.
        n (int): Number of quadrature points.

    Returns:
        points (np.ndarray): (n, D) array of quadrature points.
        weights (np.ndarray): (n, ) array of weights.
    """
    if expr_coeffs.shape[0] != 2:
        raise ValueError("expr_coeffs must have two rows")

    max_coeffs = expr_coeffs[max_idx]
    min_coeffs = expr_coeffs[min_idx]

    # Get Gauss–Legendre points and weights on [-1, 1]
    nodes, weights = leggauss(n_gl)
    weights = weights.reshape(-1,1)
    
    # Affine transformation to domain [min_coeffs, max_coeffs]
    points = 0.5 * (np.outer((nodes + 1), max_coeffs) + np.outer((1 - nodes), min_coeffs))

    # Adjust weights to match new domain
    weights = 0.5 * weights@(max_coeffs - min_coeffs).reshape(1,-1)

    return points, weights

In [ ]:
def get_quadrature_points(solution, nq: int, t_vector: np.ndarray):
    # Augment t_vector once
    t_vector_aug = np.append(t_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        qpoints, qweights = np.polynomial.legendre.leggauss(nq)
        min, max = solution[0], solution[1]
        qps_mapped = 0.5*(max*(1+qpoints) + min*(1-qpoints))
        qws_mapped = 0.5*(max-min)*qweights
        # print(max, min, qps_mapped, qws_mapped)
        return max, min, qps_mapped, qws_mapped

    for region in solution.critical_regions:
        if region.is_inside(t_vector.reshape(-1,1)):
            coeffs = np.concatenate([region.A, region.b], axis=1)[:2, :]
            qpoints, qweights = gauss_legendre_between_bounds(expr_coeffs=coeffs, n_gl=nq)
            return coeffs[0] @ t_vector_aug, coeffs[1] @ t_vector_aug, qpoints @ t_vector_aug, qweights @ t_vector_aug

    # print(f't_vector: {t_vector}')
    # print(f'solution:{solution}')
    raise ValueError("No region found that contains the given t_vector.")

In [3]:
def calculate_stocflexibility(
    sols,
    nq: Union[int, list],
    joint_func,
    d_vector: np.ndarray = None,
    verbose: bool = True,
):
    # Validate nq if it's a list
    if isinstance(nq, list):
        if len(nq) != len(sols):
            raise ValueError("If nq is a list, it must have the same length as sols")

    start_time = time.perf_counter()

    def recurse(level: int, theta_prev: list, weight_prev: float) -> float:
        if level == len(sols):
            return weight_prev * joint_func(theta_prev)

        nql = nq[level] if isinstance(nq, list) else nq

        t_vector = (
            np.block([np.array(theta_prev), d_vector])
            if isinstance(d_vector, np.ndarray)
            else np.array(theta_prev)
        )

        _, _, t_points, t_weights = get_quadrature_points(
            solution=sols[level],
            nq=nql,
            t_vector=t_vector,
        )

        t_points = t_points.flatten()
        t_weights = t_weights.flatten()

        return sum(
            recurse(level + 1, theta_prev + [v], weight_prev * w)
            for v, w in zip(t_points, t_weights)
        )

    stflex = recurse(level=0, theta_prev=[], weight_prev=1.0)

    end_time = time.perf_counter()

    if verbose:
        print(f"Gaussian Legendre Stochastic Flexibility Elapsed time: {end_time - start_time:.4f} s"
        )

    return stflex

    

In [4]:
def mpformulate_theta_bounds(flex_sol, num_theta:int, theta_bounds:list, num_design:int=0, design_bounds:list=None, psi_idx:int=0, theta_m:int=0):
    A0, b0, F0 = np.empty((len(flex_sol), num_theta)), np.empty((len(flex_sol), 1)), np.empty((len(flex_sol), num_design))
    num_cr = len(flex_sol.critical_regions)
    for i, region in enumerate(flex_sol.critical_regions):
        A0[i] = region.A[psi_idx,:num_theta]
        b0[i] = -region.b[psi_idx]
        F0[i] = -region.A[psi_idx, num_theta:num_theta+num_design]
    # print(f'num_cr:{num_cr}')
    # print(f'num_theta:{num_theta}')
    # print(f'num_design:{num_design}')
    # print(f"A0: {A0}")
    # print(f"b0: {b0}")
    # print(f"F0: {F0}")
    
    c = np.hstack([np.array([-1, 1]).reshape(1, -1), np.zeros((1, 2 * (num_theta - 1 - theta_m)))]).reshape(-1,1)
    # print(f'c:{c}')
    # print(f'c.shape: {c.shape}')
    
    row1_block = np.hstack([block for i in range(theta_m, num_theta) for block in (A0[:, [i]], np.zeros((num_cr, 1)))])
    row2_block = np.hstack([block for i in range(theta_m, num_theta) for block in (np.zeros((num_cr, 1)), A0[:, [i]])])
    bound_row = np.hstack([np.array([-1, 1]).reshape(1, -1), np.zeros((1, 2 * (num_theta - 1 - theta_m)))])
    A = np.vstack([row1_block, row2_block, bound_row, -np.eye(2*(num_theta-theta_m)), np.eye(2*(num_theta-theta_m))])
    # print(f'A: {A}')
    # print(f'A.shape: {A.shape}')
    
    x_lb = np.array([val for i in range(theta_m, len(theta_bounds)) for val in [theta_bounds[i][0]] * 2])
    x_ub = np.array([val for i in range(theta_m, len(theta_bounds)) for val in [theta_bounds[i][1]] * 2])
    b = np.vstack([b0, b0, np.zeros((1,1)), -x_lb.reshape(-1,1), x_ub.reshape(-1,1)])
    # print(f'b: {b}')
    # print(f'b.shape: {b.shape}')
    
    if F0.size==0 and theta_m==0:
        # print('here')
        return A, b, c, np.array([]), np.array([]), np.array([]), np.array([]) 
    
    F = np.vstack([F0, F0, np.zeros((1,num_design)), np.zeros((4*(num_theta-theta_m), num_design))]) if num_design>0 else np.vstack([F0, F0])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')
    if theta_m > 0:
        F_lltheta = np.hstack([A0[:, [i]] for i in range(theta_m)])
        # print(f'F_lltheta: {F_lltheta}')
        # print(f'F_lltheta.shape: {F_lltheta.shape}')
        F = np.hstack([np.vstack([-F_lltheta, -F_lltheta, np.zeros((1,len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))]), F]) if F.size > 0 else np.vstack([-F_lltheta, -F_lltheta, np.zeros((1,len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')
    
    H = np.zeros((2*(num_theta-theta_m), theta_m+num_design))
    # print(f'H:{H}')
    # print(f'H.shape: {H.shape}')
    
    A_t = np.vstack([-np.eye(theta_m+num_design), np.eye(theta_m+num_design)])
    # print(f'A_t:{A_t}')
    # print(f'A_t.shape: {A_t.shape}')
    
    theta_lb = np.array([-theta_bounds[i][0] for i in range(theta_m)] + ([-j[0] for j in design_bounds] if isinstance(design_bounds, list) 
                                                                        else [])).reshape(-1, 1)
    theta_ub = np.array([theta_bounds[i][1] for i in range(theta_m)] + ([j[1] for j in design_bounds] if isinstance(design_bounds, list) 
                                                                        else [])).reshape(-1, 1)
    
    b_t = np.vstack([theta_lb, theta_ub])
    # print(f'b_t:{b_t}')
    # print(f'b_t.shape: {b_t.shape}')
    
    return A, b, c, H, A_t, b_t, F

In [5]:
def get_theta_bounds(flex_sol, numt, tbounds, numd:int=0, dbounds:list=None):
    
    theta_bound_dict = defaultdict(dict)
    prob_dict = defaultdict(dict)
    for i in range(numt):
        A, b, c, H, A_t, b_t, F = mpformulate_theta_bounds(flex_sol=flex_sol, num_theta=numt ,num_design=numd, theta_bounds=tbounds, design_bounds=dbounds, theta_m=i)
        if F.size != 0:
            prob = MPLP_Program(A=A, b=b, c=c, H=H, A_t=A_t, b_t=b_t, F=F)
            prob.process_constraints()
            solution = solve_mpqp(problem=prob, algorithm=mpqp_algorithm.geometric)
            prob_dict[f't{i}'] = prob
            theta_bound_dict[f't{i}'] = solution
        else:
            linsol = linprog(c=c, A_ub=A, b_ub=b)
            prob_dict[f't{i}'] = linsol
            theta_bound_dict[f't{i}'] = [linsol.x[1], linsol.x[0]]
            # if linsol.success:
                # print("Optimal value:", linsol.fun)
                # print("Optimal x:", linsol.x)
        print(f'Finished solving for theta{i+1}')
    probs = [p for key, p in prob_dict.items()]
    sols = [sol for key, sol in theta_bound_dict.items()]
    
    return probs, sols

In [6]:
# Bansal (2000) Illustrative Example
t_bounds=[(0,4),(0,4)]
d_bounds=[(0,5), (0,5)]
nt = len(t_bounds)
nd = len(d_bounds)

m = MPModeler()

u = m.add_var(name='u')
x = m.add_var(name='x')
z = m.add_var(name='z')

t1 = m.add_param(name='t1')
t2 = m.add_param(name='t2')
d1 = m.add_param(name='d1')
d2 = m.add_param(name='d2')
m.add_constr(2*x - 3*z + t1 - d2 == 0)
m.add_constr(x - z/2 -t1/2 +t2/2 +d1 -7*d2/2 <= u)
m.add_constr(-2*x +2*z -4*t1/3 -t2 +2*d2 +1/3<= u)
m.add_constr(-x + 5*z/2 +t1/2 -t2 -d1 +d2/2 -1 <= u)
m.add_constr(-50 <= x)
m.add_constr(-50 <= z)
m.add_constr(t_bounds[0][0] <= t1)
m.add_constr(t_bounds[1][0] <= t2)
m.add_constr(d_bounds[0][0] <= d1)
m.add_constr(d_bounds[1][0] <= d2)
m.add_constr(t1 <= t_bounds[0][1])
m.add_constr(t2 <= t_bounds[1][1])
m.add_constr(d1 <= d_bounds[0][1])
m.add_constr(d2 <= d_bounds[1][1])
m.set_objective(u)
prob = m.formulate_problem()
prob.process_constraints()
solution_flexibility = solve_mpqp(problem=prob, algorithm=mpqp_algorithm.geometric)

start_time = time.time()
prob_list, sol_list = get_theta_bounds(flex_sol=solution_flexibility, numt=nt, numd=nd, tbounds=t_bounds, dbounds=d_bounds)
end_time = time.time()
print(f'Elapsed time for solving mp problems: {end_time-start_time}')

def joint_pdf(theta:list):
    return (2/np.pi)*np.exp(-2*((theta[0]-2)**2 + (theta[1]-2)**2))

Set parameter TokenServer to value "coe-vtls1.engr.tamu.edu"
Using a found active set [0, 1, 2]
Using a found active set [6, 9, 11, 12]
Finished solving for theta1
Using a found active set [3, 7]
Finished solving for theta2
Elapsed time for solving mp problems: 0.08356165885925293


In [24]:
d_vector = np.array([0.256112, 2.875006])
#REDUCE nq
nq = 12
sf_idx_gaussian = calculate_stocflexibility(sols=sol_list, nq=nq, joint_func=joint_pdf, d_vector=d_vector)
sf_idx_smolyak = calculate_stocflexibility_smolyak(solutions=sol_list, level = 12, joint_func=joint_pdf, d_vector=d_vector)
print(f'Stochastic Flexibility Index Gaussian Legendre quadrature: {sf_idx_gaussian:.4}')
print(f'Stochastic Flexibility Index Smolyak quadrature: {sf_idx_smolyak:.4}')

Gaussian Legendre Stochastic Flexibility Elapsed time: 0.0338 s
Smolyak stochastic flexibility computed in 0.0279 seconds.
Stochastic Flexibility Index Gaussian Legendre quadrature: 0.751
Stochastic Flexibility Index Smolyak quadrature: 0.751


## SF Expression

In [8]:
def get_bounds_regions(sols: List, min_idx: int = 1, max_idx: int = 0):
    theta_bounds_list = []
    theta_regions_list = []

    for theta_sol in sols:
        min_max_list = []
        region_list = []

        for cr in theta_sol.critical_regions:
            # Store bounds
            Ab = np.concatenate([cr.A, cr.b], axis=1)[:2]
            min_max_list.append([Ab[min_idx].tolist(), Ab[max_idx].tolist()])

            # Store region constraints
            Ef = np.concatenate([cr.E, -cr.f], axis=1)
            region_array = np.array([row.tolist() for row in Ef], dtype=float)
            region_list.append(region_array)

        # Append per-theta data
        theta_bounds_list.append(np.array(min_max_list))
        theta_regions_list.append(np.array(region_list, dtype=object))  # <-- each region is a 2D array

    return theta_bounds_list, theta_regions_list


def generate_region_combos(region_sizes, n_gl):
    """Generate region index combinations based on critical region structure."""
    n_theta = len(region_sizes)
    region_combo_shape = []
    for k in range(n_theta):
        n_paths = int(np.prod(n_gl[:k])) if k > 0 else 1
        region_combo_shape.extend([range(region_sizes[k])] * n_paths)
    return list(itertools.product(*region_combo_shape))


def affine_expr(coeffs, symbols):
    return sum(c * s for c, s in zip(coeffs[:-1], symbols)) + coeffs[-1]


def normalized_lhs(ineq):
    return ineq.lhs.expand() if hasattr(ineq, 'lhs') else None

## Smolyak SF Expression

In [18]:
def compute_sf_exprs_regions_smolyak(
    theta_bounds_list,
    theta_regions_list,
    joint_pdf_expr,
    d_syms,
    theta_syms,
    level: int,
    rule: str = "gaussian",
    growth: bool = True,
):
    """
    Compute symbolic SF expressions using a Smolyak sparse-grid quadrature.

    Parameters
    ----------
    theta_bounds_list : list
        Output of get_bounds_regions(...)[0].
        theta_bounds_list[k][r] is a 2 x N array of coefficients for min/max of θ_k
        in region r.
    theta_regions_list : list
        Output of get_bounds_regions(...)[1].
        theta_regions_list[k][r] is an array of rows encoding the polyhedral
        constraints for θ_1..θ_k, d, etc. in region r.
    joint_pdf_expr : sympy.Expr
        Joint PDF as a SymPy expression in theta_syms (e.g., θ1, θ2, ...).
    d_syms : list[sympy.Symbol]
        SymPy design / disturbance symbols, e.g. [d1, d2, ...].
    theta_syms : list[sympy.Symbol]
        SymPy parameter symbols, e.g. [t1, t2, ...].
    level : int
        Smolyak level (1,2,3,...) controlling accuracy.
    rule : str, optional
        1D quadrature rule for Smolyak, passed to chaospy (e.g. "gaussian").
    growth : bool, optional
        Whether to use nested growth for rules that support nesting.

    Returns
    -------
    sf_exprs : list[sympy.Expr]
        List of symbolic SF expressions, one per region combination.
    sf_regions : list[list[sympy.Rel]]
        Matching list of region inequality sets (constraints) for each expression.
    """

    n_theta = len(theta_syms)

    # --- 1. Build Smolyak sparse grid in u-space: u_k ∈ [-1, 1] ---

    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])

    # nodes_u: shape (n_theta, n_nodes)
    # weights_expectation: weights for expectation over U
    nodes_u, weights_expectation = cp.generate_quadrature(
        order=level,
        dist=dist,
        rule=rule,
        sparse=True,
        growth=growth,
    )

    nodes_u = np.array(nodes_u).T            # (n_nodes, n_theta)
    weights_expectation = np.array(weights_expectation).flatten()

    # Convert expectation weights E[g(U)] to integral weights over [-1,1]^n
    # E[g(U)] = 1/2^n ∫ g(u) du  ⇒  ∫ g(u) du = 2^n E[g(U)]
    weights_u = (2.0 ** n_theta) * weights_expectation

    # --- 2. Region index combinations ---
    region_sizes = [bounds.shape[0] for bounds in theta_bounds_list]
    region_combos = list(itertools.product(*[range(r) for r in region_sizes]))

    sf_exprs = []
    sf_regions = []

    # --- 3. Loop over region combinations ---
    for region_combo in region_combos:

        sf_sum = 0
        all_constraints = []

        # Loop over Smolyak nodes in u-space
        for u_vec, w_u in zip(nodes_u, weights_u):

            theta_vals = []
            jacobian = 1
            node_constraints = []

            # Nested mapping θ_k(u_k; θ_<k>, d)
            for level_idx, region_idx in enumerate(region_combo):

                # Bounds for θ_k in this region
                bounds = theta_bounds_list[level_idx][region_idx]

                # Inputs to affine bounds: [θ_0,...,θ_{k-1}, d_0,...,d_{nd-1}]
                bound_inputs = theta_vals + list(d_syms)

                # Compute t_min, t_max as SymPy expressions in d_syms
                t_min = affine_expr(bounds[0], bound_inputs)
                t_max = affine_expr(bounds[1], bound_inputs)

                # Map canonical u_k ∈ [-1,1] to θ_k
                u_k = float(u_vec[level_idx])
                t_k = 0.5 * (t_max - t_min) * u_k + 0.5 * (t_max + t_min)

                theta_vals.append(t_k)

                # Jacobian contribution
                jacobian *= 0.5 * (t_max - t_min)

                # Add region constraints for this level, using θ_vals (like GL version!)
                rows = theta_regions_list[level_idx][region_idx]
                for row in rows:
                    t_coeffs = row[:level_idx]
                    d_coeffs = row[level_idx:-1]
                    const = row[-1]
                    lhs = sum(c * theta_vals[i] for i, c in enumerate(t_coeffs)) + \
                          sum(c * d for c, d in zip(d_coeffs, d_syms)) + const
                    ineq = lhs <= 0
                    if not isinstance(ineq, (BooleanTrue, BooleanFalse)):
                        node_constraints.append(ineq)

            # Substitute θ(u,d) into PDF
            theta_subs = {sym: val for sym, val in zip(theta_syms, theta_vals)}
            pdf_val = joint_pdf_expr.subs(theta_subs)

            # Accumulate quadrature term
            sf_sum += w_u * jacobian * pdf_val
            all_constraints.extend(node_constraints)

        # --- Deduplicate constraints across nodes ---
        unique_constraints = []
        for c in all_constraints:
            if isinstance(c, (BooleanTrue, BooleanFalse)):
                continue
            if not any(
                normalized_lhs(c) == normalized_lhs(u)
                and type(c) == type(u)
                for u in unique_constraints
                if normalized_lhs(u) is not None
            ):
                unique_constraints.append(c)

        sf_exprs.append(sf_sum)
        sf_regions.append(sorted(unique_constraints, key=str))

    return sf_exprs, sf_regions


## Gaussian Legendre SF Expressions

In [10]:
def compute_sf_exprs_regions(
    theta_bounds_list,
    theta_regions_list,
    joint_pdf_expr,
    d_syms,
    n_gl_list,
    theta_syms
):
    n_theta = len(theta_syms)
    quad_data = [np.polynomial.legendre.leggauss(n) for n in n_gl_list]
    region_sizes = [bounds.shape[0] for bounds in theta_bounds_list]
    region_combos = generate_region_combos(region_sizes, n_gl_list)

    sf_exprs = []
    sf_regions = []

    for region_combo in region_combos:
        combo_ptr = 0
        # Initialize integration paths: (theta_vals, weight, scale, constraints)
        paths = [([], 1, 1, [])]

        for level in range(n_theta):
            xi, wi = quad_data[level]
            new_paths = []

            for theta_vals, weight, scale, constraints in paths:
                region_idx = region_combo[combo_ptr]
                combo_ptr += 1

                bound_inputs = theta_vals + list(d_syms)
                bounds = theta_bounds_list[level][region_idx]
                t_min = affine_expr(bounds[0], bound_inputs)
                t_max = affine_expr(bounds[1], bound_inputs)

                # Get level-specific region constraints
                rows = theta_regions_list[level][region_idx]
                level_constraints = []
                for row in rows:
                    t_coeffs = row[:level]
                    d_coeffs = row[level:-1]
                    const = row[-1]
                    lhs = sum(c * theta_vals[i] for i, c in enumerate(t_coeffs)) + \
                          sum(c * d for c, d in zip(d_coeffs, d_syms)) + const
                    ineq = lhs <= 0
                    # level_constraints.append(sp.simplify(lhs <= 0))
                    if not isinstance(ineq, (BooleanTrue, BooleanFalse)):
                        level_constraints.append(ineq)

                new_constraints = constraints + level_constraints

                # Quadrature expansion for this level
                for q in range(len(xi)):
                    t = 0.5 * (t_max - t_min) * xi[q] + 0.5 * (t_max + t_min)
                    # new_theta_vals = theta_vals + [sp.simplify(t)]
                    new_theta_vals = theta_vals + [t]
                    new_weight = weight * wi[q]
                    new_scale = scale * 0.5 * (t_max - t_min)
                    new_paths.append((new_theta_vals, new_weight, new_scale, new_constraints))

            paths = new_paths

        # Final integration and region collection
        sf_sum = 0
        all_constraints = []
        for theta_vals, weight, scale, constraints in paths:
            theta_subs = {sym: val for sym, val in zip(theta_syms, theta_vals)}
            pdf_val = joint_pdf_expr.subs(theta_subs)
            sf_sum += weight * scale * pdf_val
            all_constraints.extend(constraints)

        # Deduplicate constraints symbolically
        unique_constraints = []
        for c in all_constraints:
            if isinstance(c, (BooleanTrue, BooleanFalse)):
                print(f'Skipping trivial constraint: {c}')
            if not any(normalized_lhs(c) == normalized_lhs(u) and type(c) == type(u) for u in unique_constraints if normalized_lhs(u) is not None):
                unique_constraints.append(c)

        sf_exprs.append(sf_sum)
        # sf_regions.append(sorted(all_constraints, key=str))
        # sf_exprs.append(sp.simplify(sf_sum))
        sf_regions.append(sorted(unique_constraints, key=str))

    return sf_exprs, sf_regions

## Pyomo Model

In [19]:
import math
def preprocess_sf_expressions(
    expr_list: List[sp.Expr],
    region_list: List[List[sp.Expr]],
) -> Tuple[List[Callable], List[List[Tuple[str, Callable, Callable]]], List[str]]:
    """
    Preprocess symbolic SF and region expressions into callable Pyomo functions.
    
    Returns:
        - sf_pyomo_funcs: List of callables that accept dict of Pyomo vars and return Pyomo expressions
        - region_pyomo_funcs: Nested list of region inequality callables (lhs, rhs)
        - var_names: List of all variable names used
    """
    assert len(expr_list) == len(region_list), "Mismatch between expressions and regions"
    sf_pyomo_funcs = []
    region_pyomo_funcs = []
    all_syms = set()

    for sf_expr, reg_exprs in zip(expr_list, region_list):
        all_syms.update(sf_expr.free_symbols)
        for reg_expr in reg_exprs:
            if not isinstance(reg_expr, sp.Rel):
                raise ValueError(f"Region expression {reg_expr} is not relational")
            all_syms.update(reg_expr.free_symbols)

    var_names = sorted(str(s) for s in all_syms)

    for sf_expr in expr_list:
        sf_func = sp.lambdify(var_names, sf_expr, modules=[{'exp': exp, 'pi': math.pi}, 'sympy'])
        sf_pyomo_funcs.append(lambda var_dict, f=sf_func: f(*[var_dict[v] for v in var_names]))

    for region_exprs in region_list:
        region_funcs = []
        for reg in region_exprs:
            lhs_func = sp.lambdify(var_names, reg.lhs, modules='sympy')
            rhs_func = sp.lambdify(var_names, reg.rhs, modules='sympy')
            region_funcs.append((reg.rel_op, lhs_func, rhs_func))
        region_pyomo_funcs.append(region_funcs)

    return sf_pyomo_funcs, region_pyomo_funcs, var_names

def embed_sf_constraints_to_model(
    instance: ConcreteModel,
    sf_pyomo_funcs: List[Callable],
    region_pyomo_funcs: List[List[Tuple[str, Callable, Callable]]],
    var_names: List[str],
    bounds_dict: Dict[str, Tuple[float, float]],
    big_m: float = 1e4,
    initial_target: float = 1.0,
):
    """
    Embed SF constraints and region logic into a Pyomo model using preprocessed functions.
    """
    if not hasattr(instance, 'generated_constraints'):
        instance.generated_constraints = ConstraintList()
    if not hasattr(instance, 'region_constraints'):
        instance.region_constraints = ConstraintList()
    if not hasattr(instance, 'region_binaries'):
        instance.region_binaries = Var(RangeSet(len(sf_pyomo_funcs)), within=Binary)

    # Create design variables
    for var_name in var_names:
        if not hasattr(instance, var_name):
            setattr(instance, var_name, Var(bounds=bounds_dict[var_name]))

    # Create mapping for lambdified expressions
    var_dict = {name: getattr(instance, name) for name in var_names}

    # Mutable Param for target
    if not hasattr(instance, 'sf_target'):
        instance.sf_target = Param(mutable=True, initialize=initial_target)

    # Create sf_var and constraint
    if not hasattr(instance, 'sf'):
        instance.sf = Var(within=NonNegativeReals)

    # Store SF expressions
    sf_expr_pyomo_list = []

    for i, sf_func in enumerate(sf_pyomo_funcs):
        sf_expr = sf_func(var_dict)
        sf_expr_pyomo_list.append(sf_expr)

        # Big-M constraint: sf_expr >= target - M(1 - δ)
        delta = instance.region_binaries[i + 1]
        instance.generated_constraints.add(
            sf_expr >= instance.sf_target - big_m * (1 - delta)
        )

    # Link to overall sf var
    if not hasattr(instance, 'sf_con'):
        instance.sf_con = Constraint(
            expr=instance.sf == sum(sf_expr_pyomo_list[i] * instance.region_binaries[i + 1] for i in range(len(sf_expr_pyomo_list)))
        )

    # Region constraints
    for i, region_funcs in enumerate(region_pyomo_funcs):
        delta = instance.region_binaries[i + 1]
        for rel_op, lhs_func, rhs_func in region_funcs:
            lhs = lhs_func(*[var_dict[v] for v in var_names])
            rhs = rhs_func(*[var_dict[v] for v in var_names])
            if rel_op == '<=':
                instance.region_constraints.add(lhs <= rhs + big_m * (1 - delta))
            elif rel_op == '<':
                instance.region_constraints.add(lhs <= rhs - 1e-6 + big_m * (1 - delta))
            elif rel_op == '>=':
                instance.region_constraints.add(lhs >= rhs - big_m * (1 - delta))
            elif rel_op == '>':
                instance.region_constraints.add(lhs >= rhs + 1e-6 - big_m * (1 - delta))
            elif rel_op == '==':
                instance.region_constraints.add(lhs >= rhs - big_m * (1 - delta))
                instance.region_constraints.add(lhs <= rhs + big_m * (1 - delta))
            else:
                raise NotImplementedError(f"Unsupported operator: {rel_op}")

    # Exclusivity
    instance.region_exclusivity = Constraint(expr=sum(instance.region_binaries[i + 1] for i in range(len(sf_pyomo_funcs))) == 1)


In [ ]:
theta_bounds_list, theta_regions_list = get_bounds_regions(sols=sol_list)

theta_syms = sp.symbols(f'theta_0:{nt}')
d_syms = sp.symbols(f'd0:{nd}')
theta_0, theta_1 = theta_syms
d1, d2 = d_syms

# Reduce n_gl_list [1,1]
n_gl_list = [8,8]

joint_pdf_expr = (2/sp.pi) * sp.exp(-2 * ((theta_0 - 2) ** 2 + (theta_1 - 2) ** 2))

start_gl = time.perf_counter()
sf_exprs_gl, sf_regions_gl = compute_sf_exprs_regions(
    theta_bounds_list=theta_bounds_list,
    theta_regions_list=theta_regions_list,
    joint_pdf_expr=joint_pdf_expr,
    d_syms=d_syms,
    n_gl_list=n_gl_list,
    theta_syms=theta_syms,
)
end_gl = time.perf_counter()
time_gl = end_gl - start_gl
print(f"Gauss–Legendre symbolic SF build time: {time_gl:.4f} s")



Gauss–Legendre symbolic SF build time: 2585.9809 s
Smolyak symbolic SF build time:       1.4994 s
Smolyak / Gauss–Legendre time ratio: 0.001
Number of Smolyak SF expressions      : 6
Number of Smolyak critical regions    : 6
Number of Gaussian Legendre SF expressions : 13122
Number of Gaussian Legendre critical regions: 13122


In [22]:
# --- Smolyak symbolic SF ---
smolyak_level = 12      # or whatever level you want
smolyak_rule = "gaussian"  # or "clenshaw_curtis", etc.

start_sm = time.perf_counter()
sf_exprs_sm, sf_regions_sm = compute_sf_exprs_regions_smolyak(
    theta_bounds_list=theta_bounds_list,
    theta_regions_list=theta_regions_list,
    joint_pdf_expr=joint_pdf_expr,
    d_syms=d_syms,
    theta_syms=theta_syms,
    level=smolyak_level,
    rule=smolyak_rule,
    growth=True,
)
end_sm = time.perf_counter()
time_sm = end_sm - start_sm
print(f"Smolyak symbolic SF build time:       {time_sm:.4f} s")

# Optional: simple speedup metric
if time_sm > 0:
    print(f"Smolyak / Gauss–Legendre time ratio: {time_sm/time_gl:.3f}")

print(f'Number of Smolyak SF expressions      : {len(sf_exprs_sm)}')
print(f'Number of Smolyak critical regions    : {len(sf_regions_sm)}')
print(f'Number of Gaussian Legendre SF expressions : {len(sf_exprs_gl)}')
print(f'Number of Gaussian Legendre critical regions: {len(sf_regions_gl)}')

Smolyak symbolic SF build time:       113.1608 s
Smolyak / Gauss–Legendre time ratio: 0.044
Number of Smolyak SF expressions      : 6
Number of Smolyak critical regions    : 6
Number of Gaussian Legendre SF expressions : 13122
Number of Gaussian Legendre critical regions: 13122


In [16]:


design_bounds = {f'd{i}':bounds for i, bounds in enumerate(d_bounds)}

sf_funcs_gl, region_funcs_gl, var_names_gl = preprocess_sf_expressions(expr_list=sf_exprs_gl, region_list=sf_regions_gl)
m_sf_gl = ConcreteModel()
embed_sf_constraints_to_model(instance=m_sf_gl, sf_pyomo_funcs=sf_funcs_gl, region_pyomo_funcs=region_funcs_gl, var_names=var_names_gl, bounds_dict=design_bounds)
m_sf_gl.obj = Objective(expr=10*m_sf_gl.d0 - 10*m_sf_gl.d1, sense=minimize)
sf_target = 0.95
m_sf_gl.sf_target.set_value(sf_target)
results = SolverFactory('gams', solver='baron').solve(m_sf_gl, tee=True)

--- Job model.gms Start 12/05/25 17:46:51 45.7.0 64fbf3ce WEX-WEI x86 64bit/MS Windows
--- Applying:
    C:\GAMS\45\gmsprmNT.txt
--- GAMS Parameters defined
    Input C:\Users\SWAMIN~1.SUN\AppData\Local\Temp\tmp3lwcn_4y\model.gms
    Output C:\Users\SWAMIN~1.SUN\AppData\Local\Temp\tmp3lwcn_4y\output.lst
    ScrDir C:\Users\SWAMIN~1.SUN\AppData\Local\Temp\tmp3lwcn_4y\225a\
    SysDir C:\GAMS\45\
    CurDir C:\Users\SWAMIN~1.SUN\AppData\Local\Temp\tmp3lwcn_4y\
    LogOption 3
Licensee: MUD - 30 User License                          G230830|0002AO-GEN
          Texas A&M University, Chemical Engineering                DC11194
          C:\GAMS\45\gamslice.txt
          License Admin: Jeff Polasek, j-polasek@tamu.edu                  
          The maintenance period of the license expired on Jun 25, 2024
          Please contact GAMS or your distributor for further information
Processor information: 1 socket(s), 16 core(s), and 24 thread(s) available
GAMS 45.7.0   Copyright (C) 1987-2024 

In [23]:
sf_funcs_sm, region_funcs_sm, var_names_sm = preprocess_sf_expressions(expr_list=sf_exprs_sm, region_list=sf_regions_sm)
m_sf_sm = ConcreteModel()
embed_sf_constraints_to_model(instance=m_sf_sm, sf_pyomo_funcs=sf_funcs_sm,
                                region_pyomo_funcs=region_funcs_sm,
                                var_names=var_names_sm,
                                bounds_dict=design_bounds)
m_sf_sm.obj = Objective(expr=10*m_sf_sm.d0 - 10*m_sf_sm.d1, sense=minimize)
sf_target = 0.95
m_sf_sm.sf_target.set_value(sf_target)
results = SolverFactory('gams', solver='baron').solve(m_sf_sm, tee=True)

--- Job model.gms Start 12/05/25 18:09:34 45.7.0 64fbf3ce WEX-WEI x86 64bit/MS Windows
--- Applying:
    C:\GAMS\45\gmsprmNT.txt
--- GAMS Parameters defined
    Input C:\Users\SWAMIN~1.SUN\AppData\Local\Temp\tmpm_mrhkxn\model.gms
    Output C:\Users\SWAMIN~1.SUN\AppData\Local\Temp\tmpm_mrhkxn\output.lst
    ScrDir C:\Users\SWAMIN~1.SUN\AppData\Local\Temp\tmpm_mrhkxn\225a\
    SysDir C:\GAMS\45\
    CurDir C:\Users\SWAMIN~1.SUN\AppData\Local\Temp\tmpm_mrhkxn\
    LogOption 3
Licensee: MUD - 30 User License                          G230830|0002AO-GEN
          Texas A&M University, Chemical Engineering                DC11194
          C:\GAMS\45\gamslice.txt
          License Admin: Jeff Polasek, j-polasek@tamu.edu                  
          The maintenance period of the license expired on Jun 25, 2024
          Please contact GAMS or your distributor for further information
Processor information: 1 socket(s), 16 core(s), and 24 thread(s) available
GAMS 45.7.0   Copyright (C) 1987-2024 